# SMS Spam Detector

End-to-end walkthrough: data loading → EDA → feature engineering → model training → evaluation → error analysis.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, roc_curve, roc_auc_score,
    f1_score, precision_score, recall_score, accuracy_score,
)

from features import MessageFeatureExtractor, build_preprocessor, to_dataframe

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
print('Imports OK')

## 1. Data Loading

In [ ]:
df = pd.read_csv(
    'data/SMSSpamCollection', sep='\t', header=None,
    names=['label', 'message'], encoding='latin-1'
)
print(f'Shape: {df.shape}')
print(f'Labels: {df["label"].value_counts().to_dict()}')
df.head()

## 2. EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#388e3c', '#d32f2f'])
axes[0].set_title('Class distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center')

pct = counts / counts.sum() * 100
axes[1].pie(pct, labels=[f'{l} ({p:.1f}%)' for l, p in zip(pct.index, pct)],
            colors=['#388e3c', '#d32f2f'], startangle=90)
axes[1].set_title('Class proportions')
plt.tight_layout()
plt.show()

In [ ]:
df['char_length'] = df['message'].str.len()
df['word_count'] = df['message'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color in [('ham', '#388e3c'), ('spam', '#d32f2f')]:
    subset = df[df['label'] == label]['char_length']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=label)
axes[0].set_title('Character length distribution')
axes[0].set_xlabel('Characters')
axes[0].legend()

df.boxplot(column='word_count', by='label', ax=axes[1],
           boxprops=dict(color='navy'), medianprops=dict(color='red'))
axes[1].set_title('Word count by label')
axes[1].set_xlabel('Label')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, label, cmap in [(axes[0], 'ham', 'Greens'), (axes[1], 'spam', 'Reds')]:
        text = ' '.join(df[df['label'] == label]['message'])
        wc = WordCloud(width=600, height=300, background_color='white',
                       colormap=cmap, max_words=100).generate(text)
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(f'Word cloud: {label}')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('wordcloud not installed — skipping word cloud')

## 3. Feature Engineering

In [ ]:
extractor = MessageFeatureExtractor()
sample_msgs = [
    'FREE prize winner click now $$$!!!',
    'Hey, want to grab lunch today?',
    '',
]
features = extractor.transform(sample_msgs)
feat_df = pd.DataFrame(features, columns=extractor.get_feature_names_out(),
                        index=[f'msg{i+1}' for i in range(len(sample_msgs))])
feat_df.insert(0, 'message', sample_msgs)
feat_df

In [ ]:
# Show pairplot of numeric features colored by label
num_df = pd.DataFrame(
    extractor.transform(df['message'].values),
    columns=extractor.get_feature_names_out()
)
num_df['label'] = df['label'].values

g = sns.pairplot(num_df, hue='label', vars=['char_length', 'word_count',
                  'uppercase_ratio', 'has_currency'],
                  palette={'ham': '#388e3c', 'spam': '#d32f2f'}, plot_kws={'alpha': 0.3})
g.fig.suptitle('Numeric feature pairplot (sample)', y=1.01)
plt.show()

## 4. Model Training

In [ ]:
X = df['message'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Train spam rate: {(y_train == "spam").mean():.3f}')
print(f'Test  spam rate: {(y_test == "spam").mean():.3f}')

In [ ]:
# Baseline: always predict ham
y_baseline = np.full(len(y_test), 'ham')
print('=== Baseline (always-ham) ===')
print(classification_report(y_test, y_baseline))

In [ ]:
def make_pipeline(clf):
    return Pipeline([
        ('to_df', FunctionTransformer(to_dataframe, validate=False)),
        ('preprocessor', build_preprocessor()),
        ('clf', clf),
    ])

models = {
    'MultinomialNB': make_pipeline(MultinomialNB(alpha=0.1)),
    'LogisticRegression': make_pipeline(
        LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE,
                           class_weight='balanced')
    ),
    'LinearSVM': make_pipeline(
        SGDClassifier(loss='modified_huber', random_state=RANDOM_STATE,
                      class_weight='balanced', max_iter=1000)
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    spam_col = list(pipeline.classes_).index('spam')
    y_proba = pipeline.predict_proba(X_test)[:, spam_col]
    cv_scores = cross_val_score(pipeline, X_train, y_train,
                                cv=cv, scoring='f1_macro', n_jobs=-1)
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, pos_label='spam'),
        'Recall': recall_score(y_test, y_pred, pos_label='spam'),
        'F1': f1_score(y_test, y_pred, pos_label='spam'),
        'ROC-AUC': roc_auc_score((y_test == 'spam').astype(int), y_proba),
        'CV_F1': f'{cv_scores.mean():.4f} ± {cv_scores.std():.4f}',
        'y_pred': y_pred,
        'y_proba': y_proba,
    }
    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred))

In [ ]:
metrics_df = pd.DataFrame({
    name: {k: v for k, v in r.items() if k not in ('y_pred', 'y_proba')}
    for name, r in results.items()
}).T
display(metrics_df.style.highlight_max(
    subset=['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC'],
    color='#c8e6c9'
))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, result) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, result['y_pred'], labels=['ham', 'spam'])
    ConfusionMatrixDisplay(cm, display_labels=['ham', 'spam']).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.suptitle('Confusion matrices (test set)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
y_bin = (y_test == 'spam').astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Precision-Recall
for name, result in results.items():
    prec, rec, _ = precision_recall_curve(y_bin, result['y_proba'])
    axes[0].plot(rec, prec, label=f"{name} (F1={result['F1']:.3f})")
baseline_prec = y_bin.mean()
axes[0].axhline(baseline_prec, linestyle='--', color='gray', label='Baseline')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall curves')
axes[0].legend()

# ROC
for name, result in results.items():
    fpr, tpr, _ = roc_curve(y_bin, result['y_proba'])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={result['ROC-AUC']:.3f})")
axes[1].plot([0, 1], [0, 1], '--', color='gray', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC curves')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 5-fold CV score distribution
cv_data = []
for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X_train, y_train,
                             cv=cv, scoring='f1_macro', n_jobs=-1)
    for s in scores:
        cv_data.append({'Model': name, 'F1 macro': s})
cv_df = pd.DataFrame(cv_data)

plt.figure(figsize=(7, 4))
sns.violinplot(data=cv_df, x='Model', y='F1 macro',
               palette='muted', inner='box')
plt.title('5-fold cross-validation F1 macro')
plt.ylim(0.9, 1.01)
plt.tight_layout()
plt.show()

## 5. Error Analysis

In [ ]:
best_name = max(results, key=lambda n: results[n]['F1'])
best_result = results[best_name]
print(f'Best model: {best_name}  (F1={best_result["F1"]:.4f})')

y_pred_best = best_result['y_pred']
y_proba_best = best_result['y_proba']

fp_mask = (y_test == 'ham') & (y_pred_best == 'spam')
fn_mask = (y_test == 'spam') & (y_pred_best == 'ham')

fp_df = pd.DataFrame({'message': X_test[fp_mask], 'spam_proba': y_proba_best[fp_mask]})
fp_df = fp_df.sort_values('spam_proba', ascending=False).head(10).reset_index(drop=True)

fn_df = pd.DataFrame({'message': X_test[fn_mask], 'spam_proba': y_proba_best[fn_mask]})
fn_df = fn_df.sort_values('spam_proba', ascending=True).head(10).reset_index(drop=True)

print(f'\nFalse positives: {fp_mask.sum()}  (ham flagged as spam)')
print(f'False negatives: {fn_mask.sum()}  (spam missed)')

In [ ]:
print('False positives (ham messages flagged as spam):')
pd.set_option('display.max_colwidth', 100)
display(fp_df)

In [ ]:
print('False negatives (spam messages missed):')
display(fn_df)

## 6. Feature Importance (Logistic Regression)

In [ ]:
lr_pipeline = models['LogisticRegression']
feature_names = lr_pipeline.named_steps['preprocessor'].get_feature_names_out()
coefs = lr_pipeline.named_steps['clf'].coef_[0]
clean_names = [n.replace('tfidf__', '').replace('numeric__', '') for n in feature_names]

indices = np.argsort(coefs)
top_n = 20

top_spam_idx = indices[-top_n:][::-1]
top_ham_idx = indices[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].barh([clean_names[i] for i in top_spam_idx[::-1]],
             [coefs[i] for i in top_spam_idx[::-1]], color='#d32f2f')
axes[0].set_title('Top spam indicators')
axes[0].set_xlabel('Coefficient')

axes[1].barh([clean_names[i] for i in top_ham_idx],
             [coefs[i] for i in top_ham_idx], color='#388e3c')
axes[1].set_title('Top ham indicators')
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.show()

## 7. Threshold Analysis

In [ ]:
lr_result = results['LogisticRegression']
y_proba_lr = lr_result['y_proba']

thresholds = np.linspace(0.05, 0.95, 50)
precs, recs, f1s = [], [], []
for t in thresholds:
    y_pred_t = np.where(y_proba_lr >= t, 'spam', 'ham')
    precs.append(precision_score(y_test, y_pred_t, pos_label='spam', zero_division=0))
    recs.append(recall_score(y_test, y_pred_t, pos_label='spam', zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, pos_label='spam', zero_division=0))

plt.figure(figsize=(9, 5))
plt.plot(thresholds, precs, label='Precision', color='#1976d2')
plt.plot(thresholds, recs, label='Recall', color='#d32f2f')
plt.plot(thresholds, f1s, label='F1', color='#388e3c', linestyle='--')
plt.axvline(0.5, linestyle=':', color='gray', label='Default threshold')
plt.xlabel('Classification threshold')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs threshold (Logistic Regression)')
plt.legend()
plt.tight_layout()
plt.show()

best_t_idx = np.argmax(f1s)
print(f'Optimal threshold by F1: {thresholds[best_t_idx]:.2f}  '
      f'(F1={f1s[best_t_idx]:.4f}, P={precs[best_t_idx]:.4f}, R={recs[best_t_idx]:.4f})')

## 8. Save Model

In [ ]:
# Save the best model (Logistic Regression for interpretability)
lr_pipeline = models['LogisticRegression']
joblib.dump(lr_pipeline, 'model.joblib')
print('Saved model.joblib')

# Quick sanity check
loaded = joblib.load('model.joblib')
test_msgs = [
    'WIN FREE PRIZE NOW!!!',
    'Hey, are you free for lunch tomorrow?',
]
probas = loaded.predict_proba(test_msgs)
spam_col = list(loaded.classes_).index('spam')
for msg, proba in zip(test_msgs, probas):
    print(f'  [{proba[spam_col]:.3f} spam] {msg}')

## 9. Conclusions

| Question | Answer |
|---|---|
| What happens when spam is rare? | Class imbalance (87/13) is handled via `class_weight='balanced'`. Logistic Regression achieves high spam recall without sacrificing too much precision. |
| What kinds of messages does the model wrongly flag? | See false positive table above — often messages mentioning competitions, offers, or free events. |
| Is recall or precision more important? | Recall (catching all spam) is typically more important, but false positives erode trust. The threshold slider in the app lets users tune this. |
| Can the full pipeline be rerun from scratch? | Yes: `python train.py` downloads the dataset, retrains, and saves `model.joblib` reproducibly. |
| Can a stranger open the GitHub repo and use the app? | Yes: clone → `pip install -r requirements.txt` → `streamlit run app.py`. |

**Model choice:** Logistic Regression wins on interpretability and competitive F1. Its coefficients are directly inspectable (see feature importance section), making it the right choice for the deployed app's explainability panel.